# Update branch protection on `main`

**One-click notebook to set the required status checks on `main` for `the-lizardking/ict-trading-bot`.**

Run this once after S-045 PR #438 merges. The notebook calls the GitHub `/repos/{owner}/{repo}/branches/{branch}/protection` API and pins:

| Context | Status |
|---|---|
| `pytest-collect` | required |
| `secret-scan` | required |
| `ruff-lint` | required |
| `dry-run-guard` | required |
| `repo-inventory` | NOT required (advisory) |

Other workflows (`hf-cron`, `training-run`) are not PR-triggered and do not appear in the branch-protection list.

**How to run:** `Runtime → Run all`. No interaction needed.

**Required Colab Secret** (set once via `Tools → Secrets`, toggle *Notebook access* on):

| Name | What it holds |
|---|---|
| `GH_ADMIN_TOKEN` | A GitHub personal access token with `repo` + `admin:repo_hook` scope on `the-lizardking/ict-trading-bot` |

**Why a notebook instead of a `gh api` checklist:** per CLAUDE.md "Always do" → "For ANY manual VM operator step, deliver a one-click Colab notebook under `notebooks/operator/`, never a copy-paste CLI checklist."

In [ ]:
# Step 1: Load the admin token from Colab Secrets.
from google.colab import userdata

GH_ADMIN_TOKEN = userdata.get('GH_ADMIN_TOKEN')
if not GH_ADMIN_TOKEN:
    raise SystemExit(
        'GH_ADMIN_TOKEN not found in Colab Secrets. Open Tools → Secrets and add a personal\n'
        'access token with `repo` scope on the-lizardking/ict-trading-bot.'
    )
print('✅ Token loaded from Colab Secrets.')

In [ ]:
# Step 2: Inspect current branch protection (so we can see what we're changing).
import json
import urllib.request

OWNER = 'the-lizardking'
REPO = 'ict-trading-bot'
BRANCH = 'main'
API_BASE = f'https://api.github.com/repos/{OWNER}/{REPO}/branches/{BRANCH}/protection'
HEADERS = {
    'Authorization': f'token {GH_ADMIN_TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
    'User-Agent': 'ict-trading-bot-branch-protection-notebook',
}

req = urllib.request.Request(API_BASE, headers=HEADERS, method='GET')
try:
    with urllib.request.urlopen(req) as resp:
        current = json.load(resp)
    contexts_now = current.get('required_status_checks', {}).get('contexts', [])
    print(f'Current required contexts on {BRANCH}: {contexts_now}')
except urllib.error.HTTPError as exc:
    body = exc.read().decode('utf-8', errors='replace')
    print(f'⚠️  GET protection failed (HTTP {exc.code}): {body}')
    print('Continuing — likely no protection set yet, or the token is missing scope.')
    current = {}
    contexts_now = []

In [ ]:
# Step 3: PUT the new protection. The full body is required by the GitHub API
# even when only required_status_checks changes — null out the optional sections
# we don't want to manage from this notebook (PR review enforcement, signing,
# admin enforcement) so the existing values are preserved.
REQUIRED_CONTEXTS = ['pytest-collect', 'secret-scan', 'ruff-lint', 'dry-run-guard']

# Preserve existing PR-review / restrictions / admin settings if present.
review_enforce = current.get('required_pull_request_reviews')
restrictions = current.get('restrictions')
enforce_admins = current.get('enforce_admins', {}).get('enabled') if isinstance(current.get('enforce_admins'), dict) else current.get('enforce_admins')
linear_history = current.get('required_linear_history', {}).get('enabled', False) if isinstance(current.get('required_linear_history'), dict) else False
allow_force_pushes = current.get('allow_force_pushes', {}).get('enabled', False) if isinstance(current.get('allow_force_pushes'), dict) else False
allow_deletions = current.get('allow_deletions', {}).get('enabled', False) if isinstance(current.get('allow_deletions'), dict) else False

body = {
    'required_status_checks': {
        'strict': True,
        'contexts': REQUIRED_CONTEXTS,
    },
    'enforce_admins': bool(enforce_admins) if enforce_admins is not None else False,
    'required_pull_request_reviews': review_enforce,
    'restrictions': restrictions,
    'required_linear_history': linear_history,
    'allow_force_pushes': allow_force_pushes,
    'allow_deletions': allow_deletions,
}

data = json.dumps(body).encode('utf-8')
req = urllib.request.Request(API_BASE, data=data, headers=HEADERS, method='PUT')
try:
    with urllib.request.urlopen(req) as resp:
        result = json.load(resp)
    print(f'✅ Protection updated. New required contexts: {result.get("required_status_checks", {}).get("contexts")}')
except urllib.error.HTTPError as exc:
    body_resp = exc.read().decode('utf-8', errors='replace')
    raise SystemExit(f'PUT protection failed (HTTP {exc.code}): {body_resp}')

In [ ]:
# Step 4: Verify by reading back.
req = urllib.request.Request(API_BASE, headers=HEADERS, method='GET')
with urllib.request.urlopen(req) as resp:
    final = json.load(resp)
ctxs = final.get('required_status_checks', {}).get('contexts', [])
print(f'Final required contexts: {ctxs}')
expected = set(REQUIRED_CONTEXTS)
actual = set(ctxs)
if expected.issubset(actual):
    print('✅ All four required contexts present.')
else:
    print(f'⚠️  Missing: {expected - actual}')
extras = actual - expected
if extras:
    print(f'ℹ️  Extra contexts also pinned (not a problem unless you wanted them removed): {extras}')

## What this changes

After this notebook runs, every PR opened against `main` will be blocked from merging unless `pytest-collect`, `secret-scan`, `ruff-lint`, and `dry-run-guard` all pass. The fifth workflow (`repo-inventory`) keeps running on every PR but does not gate merges — its drift artifact is a signal for the next janitor sprint, not a hard gate.

If you ever need to add or remove a context (e.g. when promoting `repo-inventory` to blocking after enough PRs have observed the artifact), edit `REQUIRED_CONTEXTS` in step 3 and re-run the notebook. The notebook is idempotent.

## When this notebook is no longer needed

If GitHub adds a UI for managing branch-protection contexts that the operator finds easier than running this notebook, this file can be deleted — but until then, this is the canonical one-click path.